# 敘述性統計與資料摘要技術

## 📌 學習目標

完成本 Notebook 後，你將能夠：

1. 使用 Python 計算平均數、中位數、眾數等集中趨勢指標。
2. 使用變異數、標準差、全距、四分位距描述資料離散程度。
3. 判讀偏度、峰度，理解資料分佈形狀。
4. 使用直方圖與箱形圖進行資料摘要與離群值觀察。
5. 將敘述性統計應用於簡單的資料探索分析流程。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並建立一份模擬的學生成績資料，後續範例都會使用類似資料進行示範。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from collections import Counter

scores = np.array([62, 68, 70, 71, 73, 75, 78, 80, 82, 85, 88, 90, 92, 95, 100])

df = pd.DataFrame({
    'student_id': range(1, len(scores) + 1),
    'score': scores
})

print(df.head())
print('\n資料筆數：', len(df))


## 核心概念說明

敘述性統計的目的，是在不進行母體推論的前提下，快速整理並理解現有資料。

常見分析面向包括：

1. **集中趨勢**：描述資料中心位置，例如平均數、中位數、眾數。
2. **離散量度**：描述資料分散程度，例如變異數、標準差、全距、四分位距。
3. **分佈形狀**：描述資料是否偏斜、尾端是否厚重，例如偏度與峰度。

在實務資料分析中，平均數雖然直覺，但容易受到極端值影響；中位數與四分位距通常較能反映含離群值資料的典型狀況。


In [ ]:
# ── 示範：集中趨勢指標 ───────────────────────────────
# 這段程式碼示範如何計算算術平均、中位數、眾數、幾何平均與調和平均，並比較不同中心位置指標的意義。

import numpy as np
from scipy import stats
from collections import Counter

scores = np.array([62, 68, 70, 71, 73, 75, 78, 80, 82, 85, 88, 90, 92, 95, 100])

arithmetic_mean = np.mean(scores)
median = np.median(scores)
mode = Counter(scores).most_common(1)[0][0]
geometric_mean = stats.gmean(scores)
harmonic_mean = stats.hmean(scores)

print('算術平均：', round(arithmetic_mean, 2))
print('中位數：', median)
print('眾數：', mode, '（本資料每個分數只出現一次，因此眾數代表性有限）')
print('幾何平均：', round(geometric_mean, 2))
print('調和平均：', round(harmonic_mean, 2))


## 平均數與中位數的差異

當資料中出現極端值時，平均數會被拉高或拉低；中位數則只看排序後的中間位置，因此較不容易受到極端值影響。

例如薪資、房價、交易金額等資料常出現長尾分佈，這時只看平均數可能會誤判多數人的實際狀況。實務上常會同時觀察平均數、中位數、標準差與四分位距。


In [ ]:
# ── 示範：離散程度與離群值 ─────────────────────────────
# 這段程式碼比較原始資料與加入極端值後的統計量變化，觀察平均數、標準差與四分位距對離群值的敏感程度。

import numpy as np
import pandas as pd

scores = np.array([62, 68, 70, 71, 73, 75, 78, 80, 82, 85, 88, 90, 92, 95, 100])
scores_with_outlier = np.append(scores, 160)

def describe_array(data, name):
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1
    return {
        '資料': name,
        '平均數': round(np.mean(data), 2),
        '中位數': round(np.median(data), 2),
        '標準差': round(np.std(data), 2),
        '全距': round(np.max(data) - np.min(data), 2),
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'IQR': round(iqr, 2)
    }

summary = pd.DataFrame([
    describe_array(scores, '原始資料'),
    describe_array(scores_with_outlier, '加入極端值')
])

print(summary)


## 圖表化資料摘要

表格統計量能提供精確數值，但圖表能更快呈現分佈特徵。

常用圖表包括：

1. **直方圖**：觀察資料集中在哪些區間、是否偏斜或多峰。
2. **箱形圖**：觀察中位數、Q1、Q3、IQR、鬚與離群值。

箱形圖中的離群值通常以 `Q1 - 1.5 × IQR` 與 `Q3 + 1.5 × IQR` 作為判斷邊界。


In [ ]:
# ── 實際應用：直方圖、箱形圖與離群值偵測 ──────────────────────
# 這段程式碼使用直方圖與箱形圖呈現資料分佈，並用 IQR 規則找出可能的離群值。

import numpy as np
import matplotlib.pyplot as plt

scores = np.array([62, 68, 70, 71, 73, 75, 78, 80, 82, 85, 88, 90, 92, 95, 100, 160])

q1 = np.percentile(scores, 25)
q3 = np.percentile(scores, 75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = scores[(scores < lower_bound) | (scores > upper_bound)]

print('Q1：', q1)
print('Q3：', q3)
print('IQR：', iqr)
print('離群值下界：', lower_bound)
print('離群值上界：', upper_bound)
print('偵測到的離群值：', outliers)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(scores, bins=8, edgecolor='black')
plt.title('Histogram of Scores')
plt.xlabel('Score')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.boxplot(scores, vert=True)
plt.title('Box Plot of Scores')
plt.ylabel('Score')

plt.tight_layout()
plt.show()


## 🧪 自我測驗

請完成下方 TODO 填空，計算集中趨勢（平均數、中位數）、離散程度（標準差、IQR）與離群值邊界，並用偏度判斷資料是否右偏。


In [ ]:
# ── 🧪 自我測驗 ──────────────────────────────────
# 請完成下方 TODO 填空，計算集中趨勢、離散程度與離群值邊界，並確認輸出是否符合 Expected 註解。

import numpy as np
from scipy import stats

monthly_sales = np.array([120, 135, 128, 142, 138, 130, 136, 150, 145, 132, 500])

# TODO 1: 計算平均數
mean_sales = np.mean(monthly_sales)

# TODO 2: 計算中位數
median_sales = np.median(monthly_sales)

# TODO 3: 計算樣本標準差，提示：ddof=1 代表樣本標準差
sample_std = np.std(monthly_sales, ddof=1)

# TODO 4: 計算 Q1、Q3 與 IQR
q1 = np.percentile(monthly_sales, 25)
q3 = np.percentile(monthly_sales, 75)
iqr = q3 - q1

# TODO 5: 使用 IQR 規則找出離群值
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = monthly_sales[(monthly_sales < lower_bound) | (monthly_sales > upper_bound)]

# TODO 6: 計算偏度，觀察資料是否右偏
skewness = stats.skew(monthly_sales)

print('平均銷售額：', round(mean_sales, 2))
print('中位數銷售額：', round(median_sales, 2))
print('樣本標準差：', round(sample_std, 2))
print('IQR：', round(iqr, 2))
print('離群值：', outliers)
print('偏度：', round(skewness, 2))

# Expected: 平均銷售額明顯高於中位數，因為 500 是極端高值
# Expected: 離群值應包含 [500]
# Expected: 偏度應為正值，表示資料呈現右偏
